# SQL Analytics — Transaction Fraud Database

This notebook runs 8 analytical SQL queries against the SQLite transaction database and visualises the results. The queries are stored in `sql/` and cover patterns a risk analytics team needs day-to-day.

| Query | Description | Key SQL features |
|---|---|---|
| 01 | Fraud overview | Conditional aggregation, ratios |
| 02 | Rolling 7-day fraud rate | `ROWS BETWEEN`, cumulative sums |
| 03 | Velocity analysis | `LAG`, `LEAD`, `PARTITION BY`, date arithmetic |
| 04 | Account risk tiering | Chained CTEs, `PERCENT_RANK()` |
| 05 | Merchant exposure | `JOIN`, `RANK()` within tier |
| 06 | Time-of-day analysis | `CROSS JOIN`, uplift ratio |
| 07 | Alert queue | `NTILE`, composite scoring, SLA bucketing |
| 08 | Account cohort analysis | Cohort bucketing, fraud rate uplift |

**Prerequisites:** run `python scripts/build_db.py` once to generate `data/transactions.db`.

---

## Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

import sqlite3
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

from src.config import DATA_DIR

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

DB_PATH  = DATA_DIR / 'transactions.db'
SQL_DIR  = Path().resolve().parent / 'sql'

assert DB_PATH.exists(), (
    f'Database not found at {DB_PATH}.\n'
    'Run: python scripts/build_db.py'
)

# Helpers
def query(sql_file: str) -> pd.DataFrame:
    """Read a .sql file and return the result as a DataFrame."""
    sql = (SQL_DIR / sql_file).read_text()
    with sqlite3.connect(DB_PATH) as conn:
        return pd.read_sql_query(sql, conn)

def run(sql: str) -> pd.DataFrame:
    """Run an ad-hoc SQL string and return a DataFrame."""
    with sqlite3.connect(DB_PATH) as conn:
        return pd.read_sql_query(sql, conn)

print(f'Connected to: {DB_PATH}')
print('Tables:', run("SELECT name FROM sqlite_master WHERE type='table'")['name'].tolist())

---
## Query 01 — Fraud Overview

In [ ]:
overview = query('01_fraud_overview.sql')
overview.T.rename(columns={0: 'Value'})

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Transaction count split
fraud_n = int(overview['fraud_count'].iloc[0])
legit_n = int(overview['legitimate_count'].iloc[0])
axes[0].bar(['Legitimate', 'Fraud'], [legit_n, fraud_n],
            color=['#3b82f6', '#ef4444'], edgecolor='white', width=0.5)
for bar, val in zip(axes[0].patches, [legit_n, fraud_n]):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 40,
                 f'{val:,}', ha='center', fontweight='bold')
axes[0].set_title('Transaction Count', fontweight='bold')
axes[0].set_ylabel('Count')

# Volume split
fraud_vol = float(overview['fraud_volume_aud'].iloc[0])
legit_vol = float(overview['total_volume_aud'].iloc[0]) - fraud_vol
axes[1].pie([legit_vol, fraud_vol],
            labels=['Legitimate', 'Fraud'],
            colors=['#3b82f6', '#ef4444'],
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Transaction Volume (AUD)', fontweight='bold')

# Avg amount comparison
avg_legit = float(overview['avg_legit_amount'].iloc[0])
avg_fraud = float(overview['avg_fraud_amount'].iloc[0])
axes[2].bar(['Legitimate', 'Fraud'], [avg_legit, avg_fraud],
            color=['#3b82f6', '#ef4444'], edgecolor='white', width=0.5)
for bar, val in zip(axes[2].patches, [avg_legit, avg_fraud]):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 f'${val:,.0f}', ha='center', fontweight='bold')
axes[2].set_title('Average Transaction Amount', fontweight='bold')
axes[2].set_ylabel('AUD')

plt.suptitle('Fraud Overview', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Query 02 — Rolling 7-Day Fraud Rate

Uses `SUM() OVER (ROWS BETWEEN 6 PRECEDING AND CURRENT ROW)` to smooth daily noise.

In [ ]:
rolling = query('02_rolling_fraud_rate.sql')
rolling['txn_date'] = pd.to_datetime(rolling['txn_date'])
rolling.head()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# Daily vs 7-day rolling fraud rate
axes[0].plot(rolling['txn_date'], rolling['daily_fraud_rate_pct'],
             color='#94a3b8', linewidth=1, alpha=0.6, label='Daily fraud rate')
axes[0].plot(rolling['txn_date'], rolling['rolling_7d_fraud_rate_pct'],
             color='#ef4444', linewidth=2.5, label='7-day rolling avg')
axes[0].axhline(rolling['daily_fraud_rate_pct'].mean(), color='grey',
                linestyle='--', linewidth=1.2, label='Overall avg')
axes[0].set_ylabel('Fraud Rate (%)')
axes[0].set_title('Daily vs. 7-Day Rolling Fraud Rate', fontweight='bold')
axes[0].legend(fontsize=9)

# Daily transaction volume
axes[1].bar(rolling['txn_date'], rolling['daily_txns'],
            color='#3b82f6', alpha=0.7, width=0.8)
axes[1].set_ylabel('Transactions')
axes[1].set_title('Daily Transaction Volume', fontweight='bold')

plt.suptitle('Rolling Fraud Rate Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Peak daily fraud rate : {rolling['daily_fraud_rate_pct'].max():.1f}% "
      f"on {rolling.loc[rolling['daily_fraud_rate_pct'].idxmax(), 'txn_date'].date()}")
print(f"Cumulative fraud count: {int(rolling['cumulative_fraud_count'].iloc[-1]):,}")

---
## Query 03 — Velocity Analysis

Uses `LAG()` and `LEAD()` partitioned by account to detect rapid successive transactions — an ATO (account takeover) and CNP burst signature.

In [ ]:
velocity = query('03_velocity_analysis.sql')
print(f'{len(velocity):,} transactions with a prior transaction within 1 hour')
velocity[['transaction_id','account_id','txn_timestamp','amount',
           'minutes_since_prev_txn','amount_ratio_vs_prev',
           'multi_merchant_burst_flag','is_fraud']].head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Time since last transaction: fraud vs legit
for cls, colour, label in [(0,'#3b82f6','Legitimate'),(1,'#ef4444','Fraud')]:
    data = velocity[velocity['is_fraud']==cls]['minutes_since_prev_txn'].dropna()
    sns.kdeplot(data, ax=axes[0], color=colour, fill=True, alpha=0.35,
                linewidth=2, label=label)
axes[0].set_xlabel('Minutes Since Previous Transaction (same account)')
axes[0].set_ylabel('Density')
axes[0].set_title('Time Between Transactions', fontweight='bold')
axes[0].legend()

# Multi-merchant burst flag fraud rate
burst = velocity.groupby('multi_merchant_burst_flag')['is_fraud'].agg(['mean','count']).reset_index()
burst['mean'] *= 100
bars = axes[1].bar(
    ['No burst\n(same/no merchant)', 'Multi-merchant burst\n(<10 min, diff merchant)'],
    burst['mean'], color=['#3b82f6','#ef4444'], edgecolor='white', width=0.4
)
for bar, row in zip(bars, burst.itertuples()):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
                 f'{row.mean:.1f}%\n(n={row.count:,})', ha='center', fontsize=10)
axes[1].set_ylabel('Fraud Rate (%)')
axes[1].set_title('Fraud Rate: Multi-Merchant Burst Flag', fontweight='bold')

plt.suptitle('Velocity Analysis (LAG / LEAD)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Query 04 — Account Risk Tiering

Multi-step CTE builds a composite point score per account, then maps it to a risk tier. `PERCENT_RANK()` shows each account's position in the risk distribution.

In [ ]:
tiers = query('04_account_risk_tiering.sql')
print(f'{len(tiers):,} accounts scored')
print('\nRisk tier distribution:')
print(tiers['account_risk_tier'].value_counts().to_string())

In [ ]:
TIER_COLOURS = {'LOW':'#22c55e','MEDIUM':'#f59e0b','HIGH':'#ef4444','CRITICAL':'#7c3aed'}
tier_order   = ['LOW','MEDIUM','HIGH','CRITICAL']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Tier distribution
counts = tiers['account_risk_tier'].value_counts().reindex(tier_order, fill_value=0)
axes[0].bar(counts.index, counts.values,
            color=[TIER_COLOURS[t] for t in counts.index], edgecolor='white')
for bar, val in zip(axes[0].patches, counts.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                 f'{val:,}', ha='center', fontweight='bold')
axes[0].set_title('Accounts per Risk Tier', fontweight='bold')
axes[0].set_ylabel('Account Count')

# Risk score distribution
for tier in tier_order:
    data = tiers[tiers['account_risk_tier']==tier]['risk_score']
    if len(data) > 0:
        axes[1].hist(data, bins=15, color=TIER_COLOURS[tier],
                     alpha=0.6, label=tier, edgecolor='white')
axes[1].set_xlabel('Composite Risk Score')
axes[1].set_ylabel('Account Count')
axes[1].set_title('Risk Score Distribution', fontweight='bold')
axes[1].legend(fontsize=9)

# Confirmed fraud count by tier
fraud_by_tier = tiers.groupby('account_risk_tier')['confirmed_fraud_count'].sum().reindex(tier_order, fill_value=0)
axes[2].bar(fraud_by_tier.index, fraud_by_tier.values,
            color=[TIER_COLOURS[t] for t in fraud_by_tier.index], edgecolor='white')
axes[2].set_title('Confirmed Fraud Transactions\nby Account Risk Tier', fontweight='bold')
axes[2].set_ylabel('Fraud Transaction Count')

plt.suptitle('Account Risk Tiering (CTE Scoring)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nTop 5 highest-risk accounts:')
print(tiers[['account_id','account_risk_tier','risk_score','confirmed_fraud_count',
              'peak_velocity_1h','high_risk_country_txns']].head(5).to_string(index=False))

---
## Query 05 — Merchant Fraud Exposure

Ranks merchants by total fraud volume. `RANK() OVER (PARTITION BY risk_tier)` compares each merchant only against peers in the same risk tier.

In [ ]:
merchants = query('05_merchant_exposure.sql')
merchants.head(10)[['merchant_name','merchant_category','merchant_risk_tier',
                     'total_txns','fraud_txns','fraud_rate_pct','fraud_volume_aud']]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Top 10 by fraud volume
top10 = merchants.head(10).sort_values('fraud_volume_aud')
tier_col = {'Low':'#22c55e','Medium':'#f59e0b','High':'#ef4444'}
colours  = [tier_col.get(t,'#94a3b8') for t in top10['merchant_risk_tier']]
axes[0].barh(top10['merchant_name'], top10['fraud_volume_aud'],
             color=colours, edgecolor='white')
axes[0].set_xlabel('Fraud Volume (AUD)')
axes[0].set_title('Top 10 Merchants by Fraud Volume', fontweight='bold')

# Fraud rate by merchant category (top categories)
cat_stats = (merchants.groupby('merchant_category')
             .agg(fraud_txns=('fraud_txns','sum'), total_txns=('total_txns','sum'))
             .assign(fraud_rate=lambda x: x['fraud_txns']/x['total_txns']*100)
             .sort_values('fraud_rate', ascending=False).head(12))
cat_stats_s = cat_stats.sort_values('fraud_rate')
axes[1].barh(cat_stats_s.index, cat_stats_s['fraud_rate'],
             color='#f59e0b', edgecolor='white')
axes[1].axvline(merchants['fraud_rate_pct'].mean(), color='grey',
                linestyle='--', linewidth=1.2, label='Overall avg')
axes[1].set_xlabel('Fraud Rate (%)')
axes[1].set_title('Fraud Rate by Merchant Category', fontweight='bold')
axes[1].legend(fontsize=9)

plt.suptitle('Merchant Fraud Exposure', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Query 06 — Time-of-Day Analysis

`CROSS JOIN` with the overall average lets us compute an uplift ratio per hour in a single query pass.

In [ ]:
hourly = query('06_time_of_day_analysis.sql')

fig, ax1 = plt.subplots(figsize=(14, 4))

colours = ['#ef4444' if u > 1 else '#3b82f6' for u in hourly['fraud_rate_uplift']]
ax1.bar(hourly['hour'], hourly['fraud_rate_pct'], color=colours, edgecolor='white', width=0.8)
ax1.axhline(hourly['fraud_rate_pct'].mean(), color='grey', linestyle='--',
            linewidth=1.5, label=f"Overall avg ({hourly['fraud_rate_pct'].mean():.2f}%)")
ax1.axvspan(-0.5, 5.5,  alpha=0.06, color='navy')
ax1.axvspan(22.5, 23.5, alpha=0.06, color='navy', label='Night window (11pm–6am)')
ax1.set_xlabel('Hour of Day')
ax1.set_ylabel('Fraud Rate (%)')
ax1.set_xticks(range(24))
ax1.set_title('Fraud Rate and Transaction Volume by Hour', fontweight='bold')
ax1.legend(loc='upper left', fontsize=9)

ax2 = ax1.twinx()
ax2.plot(hourly['hour'], hourly['total_txns'], 'o--',
         color='#94a3b8', linewidth=2, markersize=5, label='Transaction volume')
ax2.set_ylabel('Transaction Count', color='#94a3b8')
ax2.tick_params(axis='y', labelcolor='#94a3b8')
ax2.legend(loc='upper right', fontsize=9)

plt.tight_layout()
plt.show()

night_hours = hourly[hourly['hour'].isin([23,0,1,2,3,4,5])]
day_hours   = hourly[~hourly['hour'].isin([23,0,1,2,3,4,5])]
print(f'Night window avg fraud rate : {night_hours["fraud_rate_pct"].mean():.2f}%')
print(f'Daytime avg fraud rate      : {day_hours["fraud_rate_pct"].mean():.2f}%')
print(f'Night uplift                : {night_hours["fraud_rate_uplift"].mean():.1f}x')

---
## Query 07 — Alert Queue

`NTILE(4)` buckets all alerts into four SLA bands. `ROW_NUMBER()` assigns a specific queue position. This is the query that feeds a real-time analyst dashboard.

In [ ]:
alerts = query('07_alert_queue.sql')
print(f'{len(alerts):,} alerts in queue')
print(f'Confirmed fraud in queue: {alerts["confirmed_fraud"].sum():,} '
      f'({alerts["confirmed_fraud"].mean():.1%})')
print()
print('Alerts by SLA band:')
print(alerts.groupby('review_sla')['confirmed_fraud'].agg(['count','sum','mean'])
      .rename(columns={'count':'alerts','sum':'fraud_confirmed','mean':'fraud_rate'})
      .assign(fraud_rate=lambda x: x['fraud_rate'].map('{:.1%}'.format))
      .to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Alert priority score distribution
axes[0].hist(alerts['alert_priority_score'], bins=20,
             color='#6366f1', edgecolor='white')
axes[0].set_xlabel('Alert Priority Score')
axes[0].set_ylabel('Alert Count')
axes[0].set_title('Alert Priority Score Distribution', fontweight='bold')

# Fraud rate by SLA band
sla_order  = ['P1 — Immediate','P2 — Same Day','P3 — Next Day','P4 — Weekly']
sla_fraud  = alerts.groupby('review_sla')['confirmed_fraud'].mean() * 100
sla_fraud  = sla_fraud.reindex(sla_order, fill_value=0)
sla_counts = alerts.groupby('review_sla').size().reindex(sla_order, fill_value=0)

bars = axes[1].bar(range(len(sla_order)), sla_fraud.values,
                   color=['#7c3aed','#ef4444','#f59e0b','#22c55e'], edgecolor='white')
for bar, (fr, cnt) in zip(bars, zip(sla_fraud.values, sla_counts.values)):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                 f'{fr:.1f}%\n(n={cnt})', ha='center', fontsize=9)
axes[1].set_xticks(range(len(sla_order)))
axes[1].set_xticklabels([s.split('—')[1].strip() for s in sla_order], fontsize=9)
axes[1].set_ylabel('Confirmed Fraud Rate (%)')
axes[1].set_title('Fraud Rate by Review SLA Band', fontweight='bold')

plt.suptitle('Alert Queue Analysis (NTILE Prioritisation)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nTop 5 priority alerts:')
print(alerts[['queue_position','review_sla','alert_priority_score','amount',
               'merchant_name','velocity_1h','high_risk_country','confirmed_fraud']]
      .head(5).to_string(index=False))

---
## Query 08 — Account Cohort Analysis

Fraud rate and spend behaviour broken down by account age. New accounts (<30 days) typically show higher fraud rates due to first-party fraud and synthetic identity patterns.

In [ ]:
cohorts = query('08_account_cohort_analysis.sql')
cohorts[['cohort_label','total_txns','fraud_txns','fraud_rate_pct',
          'avg_txn_amount','fraud_rate_uplift']]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
labels = [c.split('—')[1].strip() if '—' in c else c for c in cohorts['cohort_label']]

# Fraud rate by cohort
colours = ['#ef4444' if u > 1 else '#3b82f6' for u in cohorts['fraud_rate_uplift']]
axes[0].bar(labels, cohorts['fraud_rate_pct'], color=colours, edgecolor='white')
axes[0].axhline(cohorts['fraud_rate_pct'].mean(), color='grey', linestyle='--',
                linewidth=1.2, label='Overall avg')
axes[0].set_ylabel('Fraud Rate (%)')
axes[0].set_title('Fraud Rate by Account Age', fontweight='bold')
axes[0].tick_params(axis='x', rotation=20)
axes[0].legend(fontsize=8)

# Fraud rate uplift vs overall
bar_colours = ['#ef4444' if u > 1 else '#22c55e' for u in cohorts['fraud_rate_uplift']]
axes[1].bar(labels, cohorts['fraud_rate_uplift'], color=bar_colours, edgecolor='white')
axes[1].axhline(1.0, color='grey', linestyle='--', linewidth=1.2)
axes[1].set_ylabel('Uplift vs. Overall Average')
axes[1].set_title('Fraud Rate Uplift by Cohort', fontweight='bold')
axes[1].tick_params(axis='x', rotation=20)

# Average transaction amount by cohort
axes[2].bar(labels, cohorts['avg_txn_amount'], color='#6366f1', edgecolor='white')
axes[2].set_ylabel('Average Transaction Amount (AUD)')
axes[2].set_title('Avg Transaction Amount by Cohort', fontweight='bold')
axes[2].tick_params(axis='x', rotation=20)

plt.suptitle('Account Cohort Analysis by Age', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

new_acct = cohorts[cohorts['cohort_label'].str.contains('New')]
if len(new_acct) > 0:
    print(f"New account fraud rate : {new_acct['fraud_rate_pct'].iloc[0]:.1f}% "
          f"({new_acct['fraud_rate_uplift'].iloc[0]:.1f}x uplift vs. overall avg)")

---
## Summary

The 8 queries above demonstrate the full spectrum of SQL patterns used in bank risk analytics:

In [ ]:
print("""
SQL PATTERNS DEMONSTRATED
─────────────────────────────────────────────────────────────
Window functions    ROWS BETWEEN, PERCENT_RANK(), NTILE(),
                    ROW_NUMBER(), RANK(), LAG(), LEAD()

CTEs                Single-level (rolling), chained 3-step
                    (account tiering), correlated subqueries

Joins               INNER JOIN (transactions → merchants,
                    accounts), LEFT JOIN, CROSS JOIN

Aggregation         Conditional SUM/AVG with CASE WHEN,
                    COUNT DISTINCT, GROUP BY + HAVING

Date arithmetic     JULIANDAY() for elapsed-time calculation,
                    date bucketing, time-window filters

Analytical patterns Rolling averages, cohort analysis,
                    composite scoring, SLA bucketing,
                    uplift ratios, alert prioritisation
─────────────────────────────────────────────────────────────
""")